# 关于模型调用的invoke()的使用

## 关于invoke()的参数

### 1.1 文本参数
举例：

In [2]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model =  init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

In [6]:
response = model.invoke("介绍一下你自己")
print(response)


content='你好！我是 **DeepSeek**，一个由深度求索公司创造的 AI 助手。😊\n\n我可以帮你做很多事情，比如：\n\n- **回答问题**：学习、工作、生活、科技、历史等各类问题\n- **写作与润色**：文章、文案、邮件、总结、翻译等\n- **编程辅助**：代码编写、调试、解释、算法思路\n- **学习辅导**：概念讲解、题目分析、知识梳理\n- **信息整理**：归纳要点、制定计划、分析对比\n\n我的特点是支持中文和多语言交流，可以处理较长的上下文，也能读取你上传的文件内容（具体取决于当前平台功能）。不过我不能直接进行实时联网搜索、执行代码或访问外部系统；如果你需要最新信息，建议把相关资料发给我，我可以帮你分析整理。\n\n有什么我可以帮你的吗？' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问介绍自己。需要简洁介绍身份、能力、限制。可以说明我是DeepSeek AI助手，由深度求索公司创造，版本? 系统没给具体版本。可以一般性介绍。要避免过度细节。要清楚不能访问外部除非工具？这里没有工具。可以提支持文本、文件上传读取（如果平台支持），可处理多语言，免费？ 需要准确：我是AI助手，可以回答问题、写作、编程、学习辅导、信息整理等。知识截止？ 系统没有给，别乱说。可以提到不能直接执行代码/访问实时网络（取决于平台能力）。用中文友好。最终答案应简洁。'} response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 32, 'total_tokens': 353, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 143, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cac

### 1.2 字典列表
举例
主要三种角色：
- System:   系统角色，负责设置模型的行为和规则
- Human: 用户输入的问题
- Assistant：模型生成的回复（历史）

In [8]:
messages = [
    {"role": "system", "content": "你是一个专业的程序员"},
    {"role": "human", "content": "帮我解释一下什么是字典列表"},
]

response = model.invoke(messages)
response

AIMessage(content='# 什么是字典列表（List of Dictionaries）\n\n**字典列表**是一种嵌套数据结构：外层是一个**列表**，列表中的每个元素都是一个**字典**。\n\n用 Python 举例：\n\n```python\nstudents = [\n    {"name": "张三", "age": 25, "score": 90},\n    {"name": "李四", "age": 30, "score": 85},\n    {"name": "王五", "age": 28, "score": 95},\n]\n```\n\n可以这样理解：\n\n| 概念 | 作用 |\n|------|------|\n| 字典 `{...}` | 描述**一条**记录（一行数据），键是字段名 |\n| 列表 `[...]` | 存放**多条**记录（多行数据），保持顺序 |\n\n所以「一个字典 = 一行数据，一个列表 = 整张表」，这正好对应现实中的表格、JSON、数据库查询结果。\n\n---\n\n## 为什么它这么常用\n\n1. **可读性好**：用字段名访问，比 `row[2]` 清晰得多。\n2. **和 JSON 天然对应**：接口返回的数据几乎都是「对象数组」，即字典列表。\n3. **便于遍历、筛选、排序**：列表的方法和推导式都能直接用。\n4. **扩展灵活**：每行可以有不同的键（虽然一般不推荐）。\n\n---\n\n## 常见操作\n\n### 1. 访问元素\n\n```python\nstudents[0]["name"]        # \'张三\'\nstudents[1]["score"]       # 85\nstudents[-1]["name"]       # \'王五\'\n```\n\n### 2. 遍历\n\n```python\nfor s in students:\n    print(f\'{s["name"]} 的分数是 {s["score"]}\')\n```\n\n### 3. 筛选\n\n```python\npassed = [s for s in students if s["score"] >= 90]\n# [{\'name\': \'张三\',

举例2：涉及多轮对话

In [9]:
messages = [
    {"role": "system", "content": "你是一个专业的数学老师"},
    {"role": "human", "content": "1+1=？"},
    {"role": "assistant", "content": "1+1=2"},
    {"role": "human", "content": "我刚才问了什么问题？"},
]

response = model.invoke(messages)
print(response.content)

你刚才问的是：**“1+1=？”**


举例3:记忆，默认情况下，不传递历史对话，模型不会记住之前的对话

In [11]:
messages = [
    {"role": "system", "content": "你是一个非常可爱友好的AI助手"},
    {"role": "human", "content": "你好！我叫小谭"},
]

#第一次对话
response = model.invoke(messages)
print(response.content)

#第二次对话
messages = [
    {"role":"human","content":"你记不记得我叫什么名字？"}
]
response = model.invoke(messages)
print(response.content)

你好呀小谭！很高兴认识你～我是你的AI小助手，有什么想聊的、想问的，或者需要帮忙的，尽管告诉我哦！😊
抱歉，我不记得你叫什么名字。我无法访问或保留你的个人信息；如果你愿意，可以现在告诉我，我在这段对话里会按你希望的名字称呼你。


作为对比，传递记忆

In [ ]:
conversation = [
    {
        "role": "system",
        "content": "你是一个非常可爱友好的AI助手"
    },
    {
        "role": "human",
        "content": "你好！我叫小谭"
    }
]

#第一次对话
response1 = model.invoke(conversation)
print(response1.content)

#添加记忆
conversation.append({"role":"assistant","content":response1.content})
conversation.append({"role":"human","content":"你记不记得我叫什么名字？"})

#第二次对话
response2 = model.invoke(conversation)
print(response2.content)

你好呀，小谭！很高兴认识你～我是你的可爱AI助手，有什么想聊的、想问的，或者需要帮忙的，都可以随时告诉我哦！😊
当然记得呀，小谭！😊 你是刚刚跟我打招呼的小谭～有什么想聊的随时说哦！


### 1.3消息对象
举例

In [15]:
pip install langchain-core

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/
Note: you may need to restart the kernel to use updated packages.


In [17]:
from langchain_core.messages import SystemMessage, HumanMessage
messages = [
    SystemMessage(content="你是一个非常可爱友好的AI助手"),
    HumanMessage(content="你好！我叫小谭")
]

response = model.invoke(messages)
print(response.content)

嗨～小谭你好呀！很高兴认识你！😊 这个名字真好听～有什么我可以帮你的吗？或者只是想随便聊聊也完全没问题哦！


举例2：传递记忆

In [18]:
from langchain_core.messages import HumanMessage
messages = [
    SystemMessage(content="你是一个非常可爱友好的AI助手"),
    HumanMessage(content="你好！我叫小谭")
]

# 第一次对话
response1 = model.invoke(messages)
print(response1.content)


# 第二次对话
messages.append(HumanMessage(content="你好！我叫小谭"))
response2 = model.invoke(messages)
print(response2.content)


你好呀，小谭！很高兴认识你～ 😊 有什么我可以帮你的吗？
小谭你好呀！很高兴认识你～你的名字听起来很亲切呢！有什么我可以帮你的吗？或者想聊聊什么话题都可以哦～😊（对了，我该怎么称呼你比较合适呢？）


## 2.invoke()的返回值
举例

In [19]:
response = model.invoke([HumanMessage(content="1 + 1 + 1 = ?")])

In [21]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [22]:
print(response)

content='3' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple. Need comply. "1 + 1 + 1 = 3". final maybe concise.'} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 39, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 39}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '833b799e-23d6-4491-bdf0-466831fd1804', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1c0a3-72b4-75fb-bd97-f4be462f2268-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 26, 'total_tokens': 65, 'input

美化输出：
- pretty_print()：content.pretty_print()
- rich库

In [25]:
# 方式一
response.pretty_print()

================================== Ai Message ==================================

3


In [26]:
# 方式二
from rich import print as rprint
rprint(response)

AIMessage(
    content='3',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer simple. Need comply. "1 + 1 + 1 = 3". final maybe concise.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 26,
            'prompt_tokens': 39,
            'total_tokens': 65,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 24,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 39
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': '833b799e-23d6-4491-bdf0-466831fd1804',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a1c0a3-72b4-75fb-bd97-f4be462f2268-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 39,
        'output_tokens': 26,
        'total_tokens': 65,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 24}
    }
)

这里说的 缓存命中率就是主要看 response_metadata 里的 prompt_tokens_details

cached_tokens：表示命中缓存（即被缓存复用）的 token 数量。在这个例子中是 0。

cache_write_tokens：表示写入缓存的 token 数量（部分模型有，这里为 None）。

prompt_cache_hit_tokens：表示命中缓存的输入 token 总数（这里是 0）。

prompt_cache_miss_tokens：表示未命中缓存的输入 token 总数（这里是 39）。

缓存命中率 = cached_tokens / (cached_tokens + prompt_cache_miss_tokens)

举例2

In [37]:
response = model.invoke([HumanMessage(content="一句话介绍一下LLM")])

# 1.获取回复内容
print(response.content)

# 2.获取回复的元数据
metadata = response.response_metadata
print(metadata)
print(f"使用的模型：{metadata['model_name']}")
print(f"结束的原因：{metadata['finish_reason']}")
print(f"模型厂商：{metadata['model_provider']}")

# 3.token 详情
use = metadata.get("token_usage", {})
print(f"输入token数：{use.get('prompt_tokens')}")
print(f"输出token数：{use.get('completion_tokens')}")
print(f"总token数：{use.get('total_tokens')}")

# 4.ID
print(f"响应ID：{response.id}")




LLM（大语言模型）是一种基于海量文本训练、能够理解并生成自然语言的人工智能模型。
{'token_usage': {'completion_tokens': 70, 'prompt_tokens': 34, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 45, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 34}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '9f408eb6-31af-4d60-8e77-c48b28136a66', 'finish_reason': 'stop', 'logprobs': None}
使用的模型：deepseek-flash
结束的原因：stop
模型厂商：deepseek
输入token数：34
输出token数：70
总token数：104
响应ID：lc_run--01a1c0b4-12f8-723a-ba9b-95e0e9d41540-0
